## Setup

Notes:

to clean notebook

```sh
jupyter nbconvert --clear-output --inplace reactivity/reactivity_polars.ipynb
```

The following criteria were used to identify cysteine reactivity changes:

Data quality filters:
- Only peptides quantified in at least two replicates were considered
- Only proteins with at least two peptides were considered
- Percent control values for channels with a mean signal intensity < 5000 were dropped if the corresponding M0 value was also < 5000
- Residues that were quantified in two replicates where one of the replicates had < 1.5 fold change and the two replicates together had a CV > 0.5 were not considered reactivity changes (**passes_two_rep_variability_filter**).
- Proteins manually annotated as low quality ("red") were not considered reactivity changes

Reactivity change criteria:
- At least one cysteine within 1.5-fold of the expression value (**passes_bio_replicate_variation_filter**) or max/min ratio greater than 3
- At least one cysteines is changing less than 2-fold or max/min ratio greater than 3
- The cysteine must have a ratio to another cysteine on the same protein greater than 2-fold in at least 2 replicates (**passes_ratio_in_two_replicates**).
- If the cysteine has whole proteome expression data, the cysteine must have a ratio to the whole proteome value greater than 2. (**passes_expression_filter**)
- Proteins with fewer than five peptides and no expression data are listed as (Cys1;Cys2) and are counted as 1 reactivity change
- For proteins with two-four peptides, the max/min ratio must be greater than 2. For proteins with five+ peptides, the cysteine must have >2 fold change from the median cysteine value for that protein.  (**passes_ratio_or_median_filter**). 

In [ ]:
import json
import os
from functools import reduce
import math
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import sys

# Resolve `src.*` to THIS repo. A stale pre-migration copy of the package is pip-installed
# (editable) from Dropbox and otherwise wins whenever cwd is not the repo root.
_REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "macrophage.py").exists()
)
sys.path.insert(0, str(_REPO_ROOT))

from src.uniprot_utils import create_entry_cache, get_function, get_go_terms, build_gene_to_uniprot
from src.macrophage import CLINVAR_TXT, REPO_ROOT
from matplotlib_venn import venn2, venn3_circles, venn3
import polars as pl
import polars.selectors as cs
import seaborn as sns
import numpy as np
import matplotlib as mpl

# filepaths
experiment_dir = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/\
Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/\
02_reactivity/03_rc_analysis/reactivity_output/test/median_for_5+_peptides"
)
data_dir = experiment_dir / "input"
result_dir = experiment_dir / "output"
plot_dir = result_dir / "plots"
plot_dir.mkdir(exist_ok=True, parents=True)

# protein level variables
protein_level_cols = [
    "uniprot",
    "protein",
    "description",
    "condition",
    "wp_percent_control",
    "passes_bio_replicate_variation_filter",
    "rc_n_peptides",
    "median_percent_control_of_residues",
    "curation_color",
    "katya_curation_comments",
]
# boolean type variables
# don't convert to str until writing
binary_residue_cols = [
    "passes_expression_filter",
    "passes_ratio_or_median_filter",
    "reactivity_change",
    "passes_two_rep_variability_filter",
    "passes_ratio_in_two_replicates",
]


parameters = {
    "conditions": ["TLR1-2", "TLR3", "TLR4", "TLR7", "TLR8", "STING"],
    "control_condition": "M0",
    "median_filter_for_5+_peptides": True,
    "rc_wp_ratio_min_requirement": 1.5,
    "filter_based_on_raw_si": True,
}


with open(result_dir / "parameters.json", "w") as fp:
    json.dump(parameters, fp)


reference_database_folder= Path('/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/00_reference lists/reference_databases')


os.listdir(data_dir)

## Reactivity change algorithm

### Helper methods

In [ ]:
def convert_to_long_format(df, id_vars, values_name):
    df = (
        # convert to long format
        df.unpivot(
            index=id_vars,
            value_name=values_name,
            variable_name="sample",
        ).drop_nans(subset=values_name)
        # extract donor from sample name
        .with_columns(
            pl.col("sample").str.split("_").list.get(1).alias("donor"),
            pl.col("sample").str.split("_").list.get(0).alias("condition"),
        )
    )

    return df


def filter_using_raw_si(
    df: pl.DataFrame, raw_si: pl.DataFrame, id_vars: list, data_prefix: str
) -> pl.DataFrame:
    """Filters dataframe based on raw signal intensity"""
    # convert raw SI data and % control data to long format
    # merge them together

    grouping = [
        "sequence",
        "residue",
        "description",
        "protein",
        "donor",
        "condition",
        "uniprot",
    ]

    raw_si = (
        convert_to_long_format(raw_si, id_vars, f"{data_prefix}_raw_si")
        .group_by(grouping)
        # take mean of technical replicates
        .agg(pl.col("rc_raw_si").mean().alias("raw_si_mean_of_technical_replicates"))
        .pivot(on="condition", index=id_vars + ["donor"])
        .with_columns(
            pl.col(parameters["control_condition"])
            .lt(5000)
            .alias(f"lt_5000_si_in_{parameters["control_condition"]}")
        )
        .unpivot(
            on=parameters["conditions"] + [parameters["control_condition"]],
            index=id_vars
            + ["donor", f"lt_5000_si_in_{parameters["control_condition"]}"],
            value_name="raw_si_mean_of_technical_replicates",
            variable_name="condition",
        )
    )
    merged = df.join(other=raw_si, on=id_vars + ["donor", "condition"], how="left")

    filtered = (
        merged
        # filter by mean of technical replicates
        .filter(
            # keep values with mean SI > 5000
            (pl.col("raw_si_mean_of_technical_replicates").gt(pl.lit(5000)))
            # keep control condition values
            | (pl.col("condition").eq(pl.lit(parameters["control_condition"])))
            # keep values where the control condition has mean SI > 5000
            | (pl.col(f"lt_5000_si_in_{parameters["control_condition"]}").not_())
        ).select(
            id_vars
            + ["sample", f"{data_prefix}_percent_control", "donor", "condition"],
        )
    )
    return filtered


def filter_to_min_2_reps(
    df: pl.DataFrame, id_vars: list, data_prefix: str
) -> pl.DataFrame:
    """Filters dataframe to targets quantified in
    2 biological replicates"""
    values_name = f"{data_prefix}_percent_control"
    df = convert_to_long_format(df, id_vars, values_name)
    # filter to min two biological replicates
    return (
        df.group_by(id_vars)
        .agg(pl.col("donor").n_unique().alias("n_replicates"))
        .filter(pl.col("n_replicates") > 1)
        .drop("n_replicates")
        .join(other=df, on=id_vars)
    )


def passes_ratio_in_two_replicates(technical_reps_averaged):
    """Identify residues that have a ratio to another residue greater than 2-fold in at least two replicates
    Return polars dataframe"""

    replicate_grouping = ["uniprot", "condition", "donor"]

    # averages across all donors
    donor_averages = technical_reps_averaged.group_by(
        ["uniprot", "condition", "residue"]
    ).agg(pl.col("rc_percent_control_tech_rep_avg").mean().alias("avg_across_donors"))

    complete_grid = (
        technical_reps_averaged.group_by(["uniprot", "condition"])
        .agg(
            [
                pl.col("donor").unique().alias("donors"),
                pl.col("residue").unique().alias("residues"),
            ]
        )
        .explode("donors")
        .explode("residues")
        .rename({"donors": "donor", "residues": "residue"})
        .select(["uniprot", "condition", "donor", "residue"])
    )

    # fill missing combinations with averaged data
    filled_data = (
        complete_grid.join(
            technical_reps_averaged, on=replicate_grouping + ["residue"], how="left"
        )
        .join(donor_averages, on=["uniprot", "condition", "residue"], how="left")
        .with_columns(
            pl.when(pl.col("rc_percent_control_tech_rep_avg").is_null())
            .then(pl.col("avg_across_donors"))
            .otherwise(pl.col("rc_percent_control_tech_rep_avg"))
            .alias("rc_percent_control_tech_rep_avg")
        )
        .drop("avg_across_donors")
    )
    # take max and min % control of each cysteine, within each donor
    intermediate_df = (
        filled_data.group_by(replicate_grouping)
        .agg(
            pl.col("rc_percent_control_tech_rep_avg")
            .min()
            .alias("rc_min_percent_control_in_replicate"),
            pl.col("rc_percent_control_tech_rep_avg")
            .max()
            .alias("rc_max_percent_control_in_replicate"),
        )
        # calculate max ratio to another cysteine, for each cysteine, within each donor
        .join(other=technical_reps_averaged, on=replicate_grouping, how="right")
        .with_columns(
            pl.col("rc_percent_control_tech_rep_avg")
            .truediv("rc_max_percent_control_in_replicate")
            .log(base=2)
            .alias("up"),
            pl.col("rc_percent_control_tech_rep_avg")
            .truediv("rc_min_percent_control_in_replicate")
            .log(base=2)
            .alias("down"),
        )
        .with_columns(
            pl.when(pl.col("up").abs().gt(pl.col("down").abs()))
            .then(pl.col("up"))
            .otherwise(pl.col("down"))
            .map_elements(lambda x: 2**x)
            .alias("rc_max_ratio_to_another_cysteine")
        )
        # binary variable for 2x cutoff ratio to another cysteine
        .with_columns(
            pl.col("rc_max_ratio_to_another_cysteine")
            .gt(2)
            .alias("passes_2x_cutoff_ratio_to_another_cysteine_up"),
            pl.col("rc_max_ratio_to_another_cysteine")
            .lt(0.5)
            .alias("passes_2x_cutoff_ratio_to_another_cysteine_down"),
        )
    )
    two_out_of_three_ratio_df = (
        intermediate_df.group_by(  # .filter("passes_2x_cutoff_ratio_to_another_cysteine")
            ["uniprot", "condition", "residue"]
        )
        # count how many donors a cysteine passes the 2x cutoff in
        .agg(
            pl.col("passes_2x_cutoff_ratio_to_another_cysteine_down").sum(),
            pl.col("passes_2x_cutoff_ratio_to_another_cysteine_up").sum(),
            pl.col("rc_max_ratio_to_another_cysteine"),
        )
        .with_columns(
            pl.col("passes_2x_cutoff_ratio_to_another_cysteine_up")
            .gt(1)
            .or_(pl.col("passes_2x_cutoff_ratio_to_another_cysteine_down").gt(1))
            .alias("passes_ratio_in_two_replicates")
        )
        .select(["uniprot", "condition", "residue", "passes_ratio_in_two_replicates"])
    )

    return two_out_of_three_ratio_df, intermediate_df


def passes_two_rep_variability_filter(replicate_level, technical_reps_averaged):
    """Identify residues that were quantified in two replicates where one of the replicates had less than a 1.5 fold change
    and the two replicates together had a CV > 0.5.
    Returns polars dataframe"""
    grouping = ["uniprot", "condition", "residue"]
    two_rep_var_df = (
        replicate_level.filter(
            # only use finite values
            pl.col("rc_percent_control")
            .truediv(100)
            .log(base=2)
            .is_finite()
        )
        .group_by("uniprot", "residue")
        .agg(pl.col("donor").unique().count().alias("n_replicates"))
        .join(other=replicate_level, on=["uniprot", "residue"])
        .join(
            other=technical_reps_averaged,
            on=["uniprot", "residue", "donor", "condition"],
        )
        .group_by(grouping + ["n_replicates"])
        .agg(
            pl.col("rc_percent_control_tech_rep_avg")
            .truediv(100)
            .log(base=2)
            .abs()
            .le(math.log2(1.5))
            .any()
            .alias("not_changing_in_at_least_one_rep"),
            pl.col("rc_percent_control_tech_rep_avg")
            .max()
            .alias("max_rc_percent_control_tech_rep_avg"),
            pl.col("rc_percent_control_tech_rep_avg")
            .min()
            .alias("min_rc_percent_control_tech_rep_avg"),
        )
        .with_columns(
            pl.col("max_rc_percent_control_tech_rep_avg")
            .truediv(pl.col("min_rc_percent_control_tech_rep_avg"))
            .log(base=2)
            .alias("ratio_replciates")
        )
        .with_columns(
            pl.col("not_changing_in_at_least_one_rep")
            .and_(pl.col("n_replicates").eq(pl.lit(2)))
            .and_(pl.col("ratio_replciates").ge(math.log2(2)))
            .not_()
            .alias("passes_two_rep_variability_filter")
        )
        .select(grouping + ["passes_two_rep_variability_filter", "n_replicates"])
    )

    return two_rep_var_df


# read and format whole proteome data
wp_df = (
    filter_to_min_2_reps(
        df=pl.read_csv(data_dir / "02_combfiles_percctrl_wp.csv").drop(
            cs.contains("pepNum"), ""
        ),
        id_vars=["uniprot", "protein", "description"],
        data_prefix="wp",
    )
    .select(["uniprot", "wp_percent_control", "condition"])
    # take median across replicates
    .group_by(["uniprot", "condition"])
    .agg(pl.col("wp_percent_control").median())
)

# read and format reactivity data
residue_identifiers = ["uniprot", "sequence", "residue", "description", "protein"]
rc_df = filter_to_min_2_reps(
    df=pl.read_csv(
        data_dir / "02_combfiles_cysaggr_percctrl_rc.csv", ignore_errors=True
    ).drop("", "identifier"),
    id_vars=residue_identifiers,
    data_prefix="rc",
)
if parameters["filter_based_on_raw_si"]:
    raw_si_data = pl.read_csv(
        data_dir / "05_combfiles_forpca_cysaggr_channelratio_rc.csv"
    ).drop(["", "identifier"])

    rc_df = filter_using_raw_si(
        df=rc_df,
        raw_si=raw_si_data,
        id_vars=["uniprot", "sequence", "residue", "description", "protein"],
        data_prefix="rc",
    )

filtered_by_raw_si = rc_df.clone()

# join with whole proteome data
rc_df = rc_df.join(other=wp_df, how="left", on=["uniprot", "condition"])

# apply replicate level filters
replicate_level = rc_df.clone()
# average technical replicates
technical_reps_averaged = replicate_level.group_by(
    ["uniprot", "residue", "donor", "condition"]
).agg(pl.col("rc_percent_control").median().alias("rc_percent_control_tech_rep_avg"))
passes_ratio_in_two_replicates_df, intermediate_df = passes_ratio_in_two_replicates(
    technical_reps_averaged
)
passes_two_rep_variability_filter_df = passes_two_rep_variability_filter(
    replicate_level, technical_reps_averaged
)

residue_identifiers = residue_identifiers + ["wp_percent_control", "condition"]

# take median and cv of reactivity data
rc_df = (
    rc_df.group_by(residue_identifiers)
    .agg(
        pl.col("rc_percent_control").median(),
        pl.col("rc_percent_control").max().alias("rc_max_percent_control"),
        pl.col("rc_percent_control").min().alias("rc_min_percent_control"),
    )
    .filter(
        pl.col("condition") != parameters["control_condition"],
        pl.col("condition") != "TLR9",
    )
    .with_columns(
        pl.col("rc_percent_control")
        .truediv(pl.col("wp_percent_control"))
        .alias("rc_wp_ratio"),
    )
)

protein_level_df = (
    # find residues with reactivity ratios
    # within two-fold of the protein expression change
    rc_df.with_columns(
        pl.col("rc_wp_ratio")
        .log(base=2)
        .abs()
        .lt(np.log2(parameters["rc_wp_ratio_min_requirement"]))
        .fill_null(True)
        .alias("rc_wp_ratio_within_1.5_fold"),
        pl.col("rc_percent_control")
        .gt(50)
        .and_(pl.col("rc_percent_control").lt(200))
        .alias("rc_within_2_fold"),
        # add min absolute reactivity fold change
        # for filtering
        pl.col("rc_percent_control")
        .truediv(100)
        .log(base=2)
        .abs()
        .alias("rc_min_percent_control_log2_fc"),
    )
    .group_by(["uniprot", "condition"])
    .agg(
        pl.col("rc_min_percent_control_log2_fc").min(),
        pl.col("rc_wp_ratio_within_1.5_fold")
        .any()
        .or_(
            (
                pl.col("rc_wp_ratio")
                .gt(parameters["rc_wp_ratio_min_requirement"])
                .any()
            ).and_(
                pl.col("rc_wp_ratio")
                .lt(1 / parameters["rc_wp_ratio_min_requirement"])
                .any()
            )
        )
        .alias("passes_bio_replicate_variation_filter"),
        pl.col("rc_percent_control")
        .median()
        .alias("median_percent_control_of_residues"),
        pl.col("residue").n_unique().alias("rc_n_peptides"),
        # calculate ratio between max / min
        # reactivity ratio per protein
        pl.col("rc_percent_control").min().alias("rc_percent_control_min"),
        pl.col("rc_percent_control").max().alias("rc_percent_control_max"),
        # pl.col("rc_percent_control")
        # .max()
        # .truediv(pl.col("rc_percent_control").min())
        # .alias("rc_ratio_min_max"),
    )
)


curation_df = pl.read_csv(data_dir / "rc_curation_macrophage.csv").select(
    ["curation_color", "katya_curation_comments", "uniprot", "protein"]
)

# binary variable for 2x cutoff ratio to another cysteine

ratio_filter_limit = 10000
if parameters["median_filter_for_5+_peptides"]:
    ratio_filter_limit = 5
rc_df = (
    rc_df.join(other=protein_level_df, on=["uniprot", "condition"], how="left")
    .with_columns(
        # ratio to expression
        pl.col("rc_wp_ratio")
        .gt(2)
        .or_(pl.col("rc_wp_ratio").lt(0.5))
        .fill_null(True)
        .alias("passes_expression_filter"),
        # ratio to median of residues
        pl.col("rc_percent_control")
        .truediv(pl.col("median_percent_control_of_residues"))
        .alias("ratio_to_median_of_residues"),
    )
    # find resides 2-fold different from protein expression
    # residues without expression data automatically pass
    .with_columns(
        pl.col("rc_percent_control")
        .truediv("rc_percent_control_min")
        .log(base=2)
        .alias("up"),
        pl.col("rc_percent_control")
        .truediv("rc_percent_control_max")
        .log(base=2)
        .alias("down"),
    )
    .with_columns(
        pl.when(pl.col("up").abs().gt(pl.col("down").abs()))
        .then(pl.col("up"))
        .otherwise(pl.col("down"))
        .map_elements(lambda x: 2**x)
        .alias("rc_ratio_min_max")
    )
    # apply median or ratio filter,
    # depending on number of peptides
    .with_columns(
        pl.when(pl.col("rc_n_peptides") == 1)
        .then(pl.lit(False))
        .when(pl.col("rc_n_peptides") < ratio_filter_limit)
        .then(
            # requirement that ratio to another residue > 2
            (pl.col("rc_ratio_min_max") > 2).or_((pl.col("rc_ratio_min_max") < 0.5))
            # requirement that ratio to another residue and ratio to expression
            # are in the same direction. residues w/out expression pass
            .and_(
                (
                    (
                        (pl.col("rc_ratio_min_max") < 1).and_(pl.col("rc_wp_ratio") < 1)
                    ).or_(
                        (pl.col("rc_ratio_min_max") > 1).and_(pl.col("rc_wp_ratio") > 1)
                    )
                ).or_(pl.col("rc_wp_ratio").is_null())
            )
        )
        .otherwise(
            pl.when(pl.col("rc_min_percent_control_log2_fc") < 1).then(
                # requirement that the ratio to median is greater than two-fold
                (
                    pl.col("ratio_to_median_of_residues")
                    .gt(2)
                    .or_(pl.col("ratio_to_median_of_residues").lt(0.5))
                )
                # requirement that the ratio to median and ratio to expression are in the same direction
                # ignore if missing an expression value
                .and_(
                    (pl.col("wp_percent_control").is_null()).or_(
                        (
                            (pl.col("ratio_to_median_of_residues") > 1)
                            == (pl.col("rc_wp_ratio") > 1)
                        ).alias("result")
                    )
                )
            )
        )
        .alias("passes_ratio_or_median_filter")
    )
    .join(other=curation_df, on=["uniprot", "protein"], how="left")
    .join(
        other=passes_ratio_in_two_replicates_df,
        on=["uniprot", "condition", "residue"],
        how="left",
    )
    .join(
        other=passes_two_rep_variability_filter_df,
        on=["uniprot", "condition", "residue"],
        how="left",
    )
    # finally, find which residues pass all filters
    # and call them reactivity changes
    .with_columns(
        pl.col("passes_ratio_or_median_filter")
        .and_(
            pl.col("passes_expression_filter")
            .and_(
                pl.col("passes_bio_replicate_variation_filter").or_(
                    pl.col("rc_ratio_min_max").log(base=2).abs() > np.log2(3)
                )
            )
            .and_(
                (pl.col("rc_min_percent_control_log2_fc") < 1).or_(
                    pl.col("rc_ratio_min_max").log(base=2).abs() > np.log2(3)
                )
            )
            .and_(pl.col("curation_color").eq("red").not_())
            .and_(pl.col("passes_two_rep_variability_filter"))
            .and_(pl.col("passes_ratio_in_two_replicates"))
        )
        .alias("reactivity_change")
    )
    .with_columns(
        cs.exclude(protein_level_cols + binary_residue_cols)
        .cast(pl.String)
        .replace("", None)
    )
    .with_columns(
        pl.col("residue")
        .str.split("C")
        .list.get(1)
        .str.split(";")
        .list.get(0)
        .cast(pl.Int64)
        .alias("residue_num")
    )
    .sort(by="residue_num")
    .drop("residue_num")
    # replace cysteine aggregation seperator from pipeline
    # with comma TODO: do earlier in algorithm?
    .with_columns(pl.col("residue").str.replace_all(" ", "").str.replace_all(";", ","))
)

rc_df.write_csv(result_dir / "reactivity_changes_long_format.csv")

In [ ]:
protein_check = "RAB12"
conditions_check = ["TLR7", ]
filter = pl.col("protein").eq(protein_check), pl.col("condition").is_in(
    conditions_check
) #, pl.col("residue").eq("C595")

display(rc_df.filter(filter).sort(by=["condition", "residue"]))

display(replicate_level.filter(filter).sort(by=["donor", "residue"]))

## troubleshooting

In [ ]:
formatted = filtered_by_raw_si.drop(["donor", "condition"]).pivot(
    on="sample", index=["uniprot", "sequence", "residue", "description", "protein"]
)

formatted.write_csv(data_dir / "02_filtered_by_SI.csv")
#.filter(pl.col("protein").eq(pl.lit("TRIO")))

In [ ]:
print(set(pl.read_csv(data_dir / "02_combfiles_cysaggr_percctrl_rc.csv").columns).difference(set(pl.read_csv(data_dir / "05_combfiles_forpca_cysaggr_channelratio_rc.csv").columns)))


print(set(pl.read_csv(data_dir / "05_combfiles_forpca_cysaggr_channelratio_rc.csv").columns).difference(set(pl.read_csv(data_dir / "02_combfiles_cysaggr_percctrl_rc.csv").columns)))

## Formatting

In [ ]:
rc_df = rc_df.drop(
    ["rc_max_percent_control", "rc_min_percent_control", "rc_min_percent_control_log2_fc"]
)

In [ ]:
# criteria for ambiguous reactivity changes to label
# as "one" reactivity change
aggregate_criteria = (
    pl.col("wp_percent_control").is_null(),
    pl.col("rc_n_peptides") < 5,
    pl.col("reactivity_change").list.eval(pl.element().is_not_null()).list.sum() > 0,
)

# list the residue name instead of true/false
for col in binary_residue_cols:
    rc_df = rc_df.with_columns(
        pl.when(pl.col(col)).then(pl.col("residue")).otherwise(pl.lit(None)).alias(col)
    )

aggregated_df = (
    # convert df to one-protein-per-line format
    rc_df
    #.drop("n_replicates", "passes_two_rep_variability_filter", "passes_ratio_in_two_replicates")
    .group_by(protein_level_cols)
    .agg(pl.all())
    # consider ambiguous reactivity cases as "one" reactivity change
    .with_columns(
        pl.when(aggregate_criteria)
        .then(
            pl.concat_str(
                pl.lit("("),
                pl.col("reactivity_change").list.join(";"),
                pl.lit(")"),
            ).reshape((-1, 1))
        )
        .otherwise(pl.col("reactivity_change"))
        .alias("reactivity_change")
    )
    # count number of reactivity changes
    .with_columns(
        pl.col("reactivity_change")
        .list.eval(pl.element().is_not_null())
        .list.sum()
        .alias("num_reactivity_changes")
    )
    .with_columns(
        cs.exclude(protein_level_cols, "num_reactivity_changes")
        .list.eval(pl.element().cast(pl.String), parallel=True)
        .list.join("|")
    )
    .drop(["curation_color", "katya_curation_comments"])
)

identifier_cols = [
    "uniprot",
    "rc_n_peptides",
    "description",
    "protein",
    "residue",
    "sequence",
]

condition_dfs = []
for condition in parameters["conditions"]:
    df = (
        aggregated_df.filter(pl.col("condition") == condition)
        .drop(pl.col("condition"))
        .with_columns(
            cs.exclude(identifier_cols).name.prefix(f"{condition}_"),
        )
        .select(cs.contains(condition) | cs.by_name(identifier_cols))
    )
    condition_dfs.append(df)


formatted_df = reduce(
    lambda left, right: left.join(right, on=identifier_cols, how="inner"), condition_dfs
).join(
    curation_df,
    on=["uniprot", "protein"],
    how="left",
)

# Visualization

In [ ]:
# now that multiple residues count as one rc, these need to be made from a protein level df
ax = sns.barplot(
    data=aggregated_df.group_by(pl.col("condition")).agg(
        pl.col("num_reactivity_changes").sum()
    ),
    x="condition",
    y="num_reactivity_changes",
    order=parameters["conditions"],
)

ax.bar_label(ax.containers[0], fontsize=10)
plt.ylabel("number of reactivity changes")
plt.savefig(plot_dir / "reactivity_changes_barplot.pdf")
plt.show()

sns.histplot(
    data=(
        aggregated_df.with_columns(
            pl.when(pl.col("rc_n_peptides") > 5)
            .then(pl.lit("6+"))
            .otherwise(pl.col("rc_n_peptides"))
            .alias("n_peptides")
        )
        .group_by("uniprot", "n_peptides")
        .agg((pl.col("num_reactivity_changes") != 0).any().alias("reactivity_change"))
        .sort(by="n_peptides")
    ),
    x="n_peptides",
    hue="reactivity_change",
    binwidth=1,
    multiple="stack",
    palette=["grey", "red"],
)
plt.xlabel("quantified peptides")
plt.ylabel("number of proteins")
plt.savefig(plot_dir / "n_peptide_barplot.pdf")
plt.show()

## Add metadata

In [ ]:
# uniprot function annotation
cache = create_entry_cache(formatted_df, cache_path="cache.json")

formatted_df = formatted_df.with_columns(
    pl.col("uniprot")
    .map_elements(lambda x: get_function(x, cache))
    .alias("uniprot_function")
)


tiff_curation = pl.read_excel(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250627_variability_filter/median_for_5+_peptides/output/reactivity_changes_EV_macro_median_20250701_TZ.xlsx"
).select(["uniprot", "New Curation", "tiff comments"])

formatted_df = formatted_df.join(other=tiff_curation, on="uniprot", how="left")

katya_new_curation = pl.read_excel(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250627_variability_filter/median_for_5+_peptides/output/reactivity_changes_EV_macro_median_20250701_TZ_EV.xlsx"
).select(["uniprot", "curation_color_new EV", "katya_curation_comments_updated"])

formatted_df = formatted_df.join(other=katya_new_curation, on="uniprot", how="left")


eight_one_curation = pl.read_excel(
    '/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250728_no_expression_handling/median_for_5+_peptides/output/reactivity_changes_EV_0728_v3.xlsx',
    sheet_name = "Sheet1"
).select(["uniprot","comments_0801","curation_0801"])

formatted_df = formatted_df.join(other=eight_one_curation, on="uniprot", how="left")

nine_twenty_seven = pl.read_excel(
    '/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250808_two_replicate_var_and_two_out_of_three_filters/median_for_5+_peptides/output/reactivity_changes_Macrophages_20250808_filters_EV_HS.xlsx',
    sheet_name = "other"
).select(["uniprot","comments_0801","curation 0808",	"COMMENTS_HS_0912"])

formatted_df = formatted_df.join(other=nine_twenty_seven, on="uniprot", how="left")

In [ ]:
# compare to April 29 output
output_0429 = "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250429_3reps_new filters/rc_df_20250429_new filters_Macrophages_EV.xlsx"

df = (
    pl.read_excel(
        output_0429,
    )
    .select(
        "uniprot",
        cs.contains("num_reactivity_changes"),
    )
    .join(
        # reading a different excel sheet
        other=pl.read_excel(output_0429, sheet_name="Sheet1").select(
            "uniprot", "katya_curation_comments", "katya_comments", "process"
        ),
        on="uniprot",
        how="left",
    )
    .with_columns(cs.contains("num_reactivity_changes").name.prefix("0429_output_"))
    .drop(cs.starts_with("num_reactivity_changes"))
)

merged = formatted_df.join(df, on="uniprot")
for condition in parameters["conditions"]:
    merged = merged.with_columns(
        pl.col(f"{condition}_num_reactivity_changes")
        .eq(pl.col(f"0429_output_num_reactivity_changes_{condition}"))
        .alias(f"{condition}_num_reactivity_changes_matches_0429_output")
    )
merged.sort(by = "TLR4_num_reactivity_changes").write_csv(result_dir/"reactivity_changes.csv")

In [ ]:
# compare to June 27 output

output_0627 = "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250627_variability_filter/median_for_5+_peptides/output/reactivity_changes.csv"


df = (
    pl.read_csv(output_0627)
    .select("uniprot", cs.contains("num_reactivity_changes") - cs.contains("0429"))
    .with_columns(cs.contains("num_reactivity_changes").name.prefix("0627_output_"))
    .drop(cs.contains("num_reactivity_changes") - cs.contains("0627"))
)

merged_0627 = merged.join(df, on="uniprot")
for condition in parameters["conditions"]:
    merged_0627 = merged_0627.with_columns(
        pl.col(f"{condition}_num_reactivity_changes")
        .eq(pl.col(f"0627_output_{condition}_num_reactivity_changes"))
        .alias(f"{condition}_num_reactivity_changes_matches_0627_output")
    )
# merged_0627.write_csv(result_dir / "reactivity_changes.csv")

output_0728 = "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250728_no_expression_handling/median_for_5+_peptides/output/reactivity_changes.csv"

df = (
    pl.read_csv(output_0728)
    .select(
        "uniprot",
        cs.contains("num_reactivity_changes")
        - cs.contains("0429")
        - cs.contains("0627"),
    )
    .with_columns(cs.contains("num_reactivity_changes").name.prefix("0728_output_"))
    .drop(cs.contains("num_reactivity_changes") - cs.contains("0728"))
)

merged_0728 = merged_0627.join(df, on="uniprot")
for condition in parameters["conditions"]:
    merged_0728 = merged_0728.with_columns(
        pl.col(f"{condition}_num_reactivity_changes")
        .eq(pl.col(f"0728_output_{condition}_num_reactivity_changes"))
        .alias(f"{condition}_num_reactivity_changes_matches_0728_output")
    )
merged_0728.sort(by="TLR4_num_reactivity_changes", descending=True).write_csv(
    result_dir / "reactivity_changes.csv"
)



output_0801 = '/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250808_two_replicate_var_and_two_out_of_three_filters/median_for_5+_peptides/output/reactivity_changes.csv'

df = (
    pl.read_csv(output_0801)
    .select(
        "uniprot",
        cs.contains("num_reactivity_changes")
        - cs.contains("0429")
        - cs.contains("0627")
        - cs.contains("0728"),
    )
    .with_columns(cs.contains("num_reactivity_changes").name.prefix("0801_output_"))
    .drop(cs.contains("num_reactivity_changes") - cs.contains("0801"))
)

merged_0801 = merged_0728.join(df, on="uniprot")
for condition in parameters["conditions"]:
    merged_0801 = merged_0801.with_columns(
        pl.col(f"{condition}_num_reactivity_changes")
        .eq(pl.col(f"0801_output_{condition}_num_reactivity_changes"))
        .alias(f"{condition}_num_reactivity_changes_matches_0801_output")
    )
merged_0801.sort(by="TLR4_num_reactivity_changes", descending=True).write_csv(
    result_dir / "reactivity_changes.csv"
)


output_0915 = '/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250915_better_two_out_of_three_filter/median_for_5+_peptides/output/reactivity_changes.csv'

df = (
    pl.read_csv(output_0915)
    .select(
        "uniprot",
        cs.contains("num_reactivity_changes")
        - cs.contains("0429")
        - cs.contains("0627")
        - cs.contains("0728")
        - cs.contains("0801"),
    )
    .with_columns(cs.contains("num_reactivity_changes").name.prefix("0915_output_"))
    .drop(cs.contains("num_reactivity_changes") - cs.contains("0915"))
)

merged_0915 = merged_0801.join(df, on="uniprot")
for condition in parameters["conditions"]:
    merged_0915 = merged_0915.with_columns(
        pl.col(f"{condition}_num_reactivity_changes")
        .eq(pl.col(f"0915_output_{condition}_num_reactivity_changes"))
        .alias(f"{condition}_num_reactivity_changes_matches_0915_output")
    )
merged_0915.sort(by="TLR4_num_reactivity_changes", descending=True).write_csv(
    result_dir / "reactivity_changes.csv"
)


output_1007 = '/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20251007_improved_two_out_of_three_filter/median_for_5+_peptides/output/reactivity_changes.csv'

df = (
    pl.read_csv(output_1007)
    .select(
        "uniprot",
        cs.contains("num_reactivity_changes")
        - cs.contains("0429")
        - cs.contains("0627")
        - cs.contains("0728")
        - cs.contains("0801")
        - cs.contains("0915"),
    )
    .with_columns(cs.contains("num_reactivity_changes").name.prefix("1007_output_"))
    .drop(cs.contains("num_reactivity_changes") - cs.contains("1007"))
)

merged_1007 = merged_0915.join(df, on="uniprot")
for condition in parameters["conditions"]:
    merged_1007 = merged_1007.with_columns(
        pl.col(f"{condition}_num_reactivity_changes")
        .eq(pl.col(f"1007_output_{condition}_num_reactivity_changes"))
        .alias(f"{condition}_num_reactivity_changes_matches_1007_output")
    )


output_1010 = '/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20251010_residue_level_ratios/median_for_5+_peptides/output/reactivity_changes.csv'

df = (
    pl.read_csv(output_1010)
    .select(
        "uniprot",
        cs.contains("num_reactivity_changes")
        - cs.contains("0429")
        - cs.contains("0627")
        - cs.contains("0728")
        - cs.contains("0801")
        - cs.contains("0915")
        - cs.contains("1007"),
    )
    .with_columns(cs.contains("num_reactivity_changes").name.prefix("1010_output_"))
    .drop(cs.contains("num_reactivity_changes") - cs.contains("1010"))
)

merged_1010 = merged_1007.join(df, on="uniprot")
for condition in parameters["conditions"]:
    merged_1010 = merged_1010.with_columns(
        pl.col(f"{condition}_num_reactivity_changes")
        .eq(pl.col(f"1010_output_{condition}_num_reactivity_changes"))
        .alias(f"{condition}_num_reactivity_changes_matches_1010_output")
    )
merged_1010.sort(by="TLR4_num_reactivity_changes", descending=True).write_csv(
    result_dir / "reactivity_changes.csv"
)

# Examining M0 signal intensities

In [ ]:
from pathlib import Path
import os
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

processed_file_dir = Path(
    "/Users/henrysanford/dev/test_data/macrophage/02_reactivity/03_rc_analysis/20250806_raw_si/01_processed_files"
)
dfs = []
for l in os.listdir(processed_file_dir):
    if "processed" in l:
        dfs.append(
            pl.read_csv(processed_file_dir / l).with_columns(
                pl.lit(l.split("_census-out.csv")[0].split("processed_")[1]).alias(
                    "experiment"
                )
            )
        )

raw_si = (
    pl.concat(items=dfs, how="vertical_relaxed")
    .with_columns(
        pl.col("tag_126.127726")
        .add(pl.col("tag_127.124761"))
        .truediv(2)
        .alias("mean_m0"),
        (
            pl.concat_list(
                [pl.col("tag_126.127726"), pl.col("tag_127.124761")]
            ).list.std()
            / pl.concat_list(
                [pl.col("tag_126.127726"), pl.col("tag_127.124761")]
            ).list.mean()
        ).alias("cv_m0"),
    )
    .sort(by="mean_m0")
    .with_columns(
        pl
        .when(pl.col("mean_m0").gt(5000))
        .then(pl.lit(">5000"))
        .otherwise(pl.lit("0-5000"))
        .alias("M0 average signal intensity"),
        pl.when(pl.col("cv_m0").lt(0.5))
        .then(pl.lit("0-0.5"))
        .otherwise(pl.lit(">0.5"))
        .alias("CV of M0 channels"),
    )
)

# Create figure with subplots side by side
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6),sharex=True)

# First histogram
sns.histplot(
    data=raw_si,
    y="experiment",
    hue="M0 average signal intensity",
    multiple="stack",
    ax=ax1,
    palette=["red", "grey"]
)
ax1.set_xlabel("peptides")
ax1.set_title("Average Signal Intensity of M0 Channels")

# Second histogram
sns.histplot(
    data=raw_si,
    y="experiment",
    hue="CV of M0 channels",
    multiple="stack",
    ax=ax2,
    palette=["red", "grey"]
)
ax2.set_xlabel("peptides")
ax2.set_title("CV of M0 Channels")

plt.tight_layout()
plt.show()

TZ2-84: 14505
TZ3-103C: 16624
TZ2-198A: 25960

# Troubleshooting

In [ ]:
new = pl.read_csv('/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250807_raw_si_filtering/median_for_5+_peptides/output/reactivity_changes_long_format.csv')

pl.read_csv(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250728_no_expression_handling/median_for_5+_peptides/output/reactivity_changes_long_format0728.csv"
).join(other = new, on = ["uniprot", "residue","condition"]).filter(pl.col("reactivity_change_right").not_() & pl.col("reactivity_change"))

# 0915 troubleshooting

In [ ]:
import polars as pl

notes = pl.read_excel(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250808_two_replicate_var_and_two_out_of_three_filters/median_for_5+_peptides/output/reactivity_changes_Macrophages_20250808_filters_EV.xlsx",
    sheet_name="other",
)

data = pl.read_csv(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/02_reactivity/03_rc_analysis/reactivity_output/20250808_two_replicate_var_and_two_out_of_three_filters/median_for_5+_peptides/output/reactivity_changes_long_format.csv"
)


data.join(
    other=notes.select(["uniprot", "curation 0808"]), on="uniprot", how="inner"
).filter(
    #pl.col("curation 0808").str.contains("TLR4"), 
    pl.col("condition").eq("TLR3"),
    pl.col("uniprot").eq("Q9Y4D7")
).sort(
    by="uniprot"
)

# Directionality

In [ ]:
pl.read_csv(result_dir / "reactivity_changes_long_format.csv").with_columns(
    pl.when(pl.col("wp_percent_control").is_null())
    .then(pl.col("median_percent_control_of_residues"))
    .otherwise(pl.col("wp_percent_control"))
    .alias("reference_percent_control")
).with_columns(
    pl.when(pl.col("reactivity_change").eq(True).not_())
    .then(pl.lit("unchanged"))
    .when(pl.col("rc_percent_control").gt(pl.col("reference_percent_control")))
    .then(pl.lit("Higher"))
    .otherwise(pl.lit("Lower"))
    .alias("direction_of_reactivity_change")
).write_csv(
    result_dir / "reactivity_changes_long_format.csv"
)

# Overlap w other studies

In [ ]:
import matplotlib
from matplotlib_venn import venn3, venn3_circles, venn3_unweighted

matplotlib.rcParams['pdf.fonttype'] = 42 
sets = {
    "Macrophage": set(
        pl.read_csv(result_dir / "reactivity_changes_long_format.csv").filter(
            pl.col("reactivity_change")
        )["uniprot"]
    ),
    "T cell exhaustion": set(
        pl.read_csv(
            "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Exhaustion manuscript/02_Mass-spectrometry data/02_reactivity/03_rc_analysis/20240821_5_rep_variability_filter/05_reactivity_changes/output/rc_df.csv",
            ignore_errors=True,
        ).filter(
            pl.any_horizontal(pl.col("^residues_reactivity_changes_.*$").is_not_null())
        )[
            "uniprot"
        ]
    ),
    "Cell 2020": set(
        pl.read_csv(
            "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Vinogradova Laboratory/Henry_data processing/papers/chemical proteomics/Cell_2020/s5.csv"
        )["Uniprot"]
    ),
}

fig, ax = plt.subplots(figsize=(3, 3))
v = venn3(
    subsets=sets.values(),
    set_labels=sets.keys(),
    ax=ax,
    set_colors=["#4869B2", "#FFCC31", "#237D41"],
)

venn3_circles(subsets=sets.values(), ax=ax, linewidth=0.25)

for text in v.set_labels:
    if text:
        text.set_fontsize(6)
        text.set_color("black")
        text.set_fontfamily("Arial")

for text in v.subset_labels:
    if text:
        text.set_fontsize(6)
        text.set_color("black")
        text.set_fontfamily("Arial")


plt.tight_layout()

#plt.savefig(plot_dir / "venn_diagram.pdf" , dpi=300)
#sets.pop("T cell exhaustion")

In [ ]:
import polars as pl

# sets: dict[str, set[str]]

# 1. Collect all unique uniprot IDs
all_items = pl.Series("uniprot", sorted(set().union(*sets.values())))

# 2. Build columns: 1 if the item is in the set, else 0
df = pl.DataFrame({"uniprot": all_items})

for name, s in sets.items():
    df = df.with_columns(
        pl.col("uniprot").is_in(list(s)).cast(pl.Int8).alias(name)
    )

print(df)


In [ ]:
pl.read_csv(result_dir / "reactivity_changes_long_format.csv").select(["uniprot","protein","uniprot_function"]).unique().join(other = df, how = "right", on = "uniprot").write_csv("/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/02_Figures/Figure 3_reactivity profiling broad/Panels/venn_diagram/venn_data.csv")

In [ ]:
from matplotlib_venn import venn2, venn2_circles

fig, ax = plt.subplots(figsize=(3, 3))
v = venn2(
    subsets=sets.values(),
    set_labels=sets.keys(),
    ax=ax,
    set_colors=["#4869B2", "#237D41"],
)

venn2_circles(subsets=sets.values(), ax=ax, linewidth=0.25)

for text in v.set_labels:
    if text:
        text.set_fontsize(6)
        text.set_color("black")
        text.set_fontfamily("Arial")

for text in v.subset_labels:
    if text:
        text.set_fontsize(6)
        text.set_color("black")
        text.set_fontfamily("Arial")


plt.tight_layout()

plt.savefig(plot_dir / "venn_diagram.pdf" , dpi=300)

# Location

In [ ]:
df = pl.read_csv(result_dir / "reactivity_changes_long_format.csv").with_columns(
    [
        pl.col("uniprot")
        .map_elements(
            lambda x: get_function(x, cache=cache),return_dtype=pl.String
        )
        .alias("uniprot_function"),
        pl.col("uniprot")
        .map_elements(
            lambda x: get_go_terms(x, cache=cache),return_dtype=pl.String
        )
        .alias("uniprot_goterms")
    ]
)

from collections import OrderedDict


"""
df = df.with_columns(
    pl.col("uniprot_goterms").map_elements(
        lambda x: next((key for key, term in go_terms_for_pie_chart.items() if term in x), None),
        return_dtype=pl.Utf8
    ).alias("location")
)

df.write_csv(result_dir / "reactivity_changes_long_format.csv")
"""

In [ ]:
df = pl.read_csv(result_dir / "reactivity_changes_long_format.csv")

proteins_rc_changes = (
    df.filter(pl.col("reactivity_change")).select("uniprot", "uniprot_goterms").unique()
)

go_terms_for_pie_chart = {
    "golgi": "GO:0005794",
    "endoplasmic reticulum": "endoplasmic reticulum",
    "mitochondria": "GO:0005739",
    "cytoplasm": "GO:0005829",
    "nucleus": "nucle",
    "endosome": "endosome",
    "lysosome": "lysosome",
}

for name, substring in go_terms_for_pie_chart.items():
    proteins_rc_changes = proteins_rc_changes.with_columns(
        pl.col("uniprot_goterms").str.contains(substring).alias(name)
    )
df_1 = proteins_rc_changes

proteins_rc_changes = proteins_rc_changes.unpivot(index=["uniprot", "uniprot_goterms"]).filter(
    pl.col("value")
).group_by("variable").agg(
    pl.col("uniprot")
    .count()
    .truediv(len(set(proteins_rc_changes["uniprot"])))
    .mul(100)
    .alias("percentage_of_reactivity_changes")
)

sns.barplot(
    data = proteins_rc_changes,
    y = "variable", x= "percentage_of_reactivity_changes"
)


proteins_rc_changes.write_csv("/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/02_Figures/Figure 3_reactivity profiling broad/Panels/location_pie_chart/percentage_of_reactivity_changes.csv")

In [ ]:
df.head(10)

# Cross referencing

In [ ]:
from Bio import SeqIO



def read_fasta(fn):
    seqs = {}
    for record in SeqIO.parse(fn, format="fasta"):
        uniprot_id = str(record.id).split("|")[1]
        seqs[uniprot_id] = str(record.seq).replace("*", "")
    return seqs


uniprot_canonical = read_fasta(
    reference_database_folder / "uniprotkb_proteome_UP000005640_AND_revi_2024_03_12.fasta"
)

# CORUM

corum = (
    pl.read_excel(
        reference_database_folder / "CORUM download 2022_09_12.xlsx"
    )
    .filter(pl.col("Organism").eq("Human"))
    .select("subunits(UniProt IDs)", "ComplexName")
    .with_columns(pl.col("subunits(UniProt IDs)").str.split(";").alias("uniprot"))
    .explode("uniprot")
    .drop("subunits(UniProt IDs)")
    .group_by("uniprot")
    .agg(pl.col("ComplexName").unique().str.concat("|"))
)

df = pl.read_csv(result_dir / "reactivity_changes_long_format.csv").join(
    other=corum, on="uniprot", how="left"
)

print(len(df))

# COMPLEXPORTAL DATA
# https://www.ebi.ac.uk/complexportal/download
complexportal = (
    pl.read_csv(reference_database_folder / "9606.tsv", separator="\t")
    .with_columns(
        pl.col("Identifiers (and stoichiometry) of molecules in complex")
        .str.split("|")
        .alias("uniprot")
    )
    .explode("uniprot")
    .with_columns(
        pl.col("uniprot").str.split("(").list.get(0).str.split("-").list.get(0)
    )
    .filter(
        # filter to proteins
        pl.col("uniprot").is_in(set(uniprot_canonical.keys()))
    )
    .select(["#Complex ac", "Recommended name", "uniprot"])
    .unique()
    .rename(
        {
            "#Complex ac": "ComplexPortal_complex",
            "Recommended name": "ComplexPortal_name",
        }
    )
    .group_by("uniprot")
    .agg(pl.col("ComplexPortal_name").unique().str.concat("|"))
)

df = df.join(other=complexportal, on="uniprot", how="left")

# CONSERVATION

# cysteine conservation data doi.org/10.1038/s41586-025-08756-y table 10
id_cols = ["Accession", "Gene", "Cysteine"]
cysteine_conservation = (
    pl.scan_csv(
        reference_database_folder / "cysteine_conservation.csv",
        skip_rows=1,
    )
    .drop_nulls()
    .with_columns(
        pl.col(id_cols).str.split(" "),
        pl.col("Depth of Conservation (# of Organisms)")
        .truediv(1.02)  # 102 organisms
        .alias("percent conservation"),
    )
    .explode(id_cols)
    .sort(by="percent conservation")
    .with_columns(pl.concat_str(pl.lit("C"), pl.col("Cysteine")).alias("first_residue"))
    .filter(pl.col("Accession").is_in(set(df["uniprot"])))
    .group_by(["Accession", "first_residue"])
    .agg(pl.all().mean())
    .with_columns(pl.col("Accession").alias("uniprot"))
    .select("uniprot", "first_residue", "Depth of Conservation (# of Organisms)")
    .collect()
)


df = df.with_columns(
    pl.col("residue").str.split(",").list.get(0).alias("first_residue")
).join(other=cysteine_conservation, on=["uniprot", "first_residue"], how="left")

print(len(df))
# SOLVENT ACCESSIBILITY
# read/prepare pPSE scores
white_2023_pse = (
    pl.read_csv(
        reference_database_folder / "20230315_Supplementary_AlphaFold_pPSE.csv",
        ignore_errors=True,
    )
    .rename({"protein_id": "uniprot", "position": "cysteine_indices"})
    .with_columns(
        pl.concat_str(pl.lit("C"), pl.col("cysteine_indices")).alias("residue")
    )
    .filter(pl.col("uniprot").is_in(df["uniprot"]))
    .select(["uniprot", "pPSE", "residue", "structure_group"])
)

df = df.join(white_2023_pse, on=["uniprot", "residue"], how="left").with_columns(
    pl.col("residue")
    .str.split("C")
    .list.get(1)
    .str.split(",")
    .list.get(0)
    .cast(pl.Int64)
    .alias("pos")
)
print(len(df))
# AM

"""
# filter AM database to proteins in our study
# to reduce filesize
pl.scan_csv(
    "/Users/henrysanford/Downloads/AlphaMissense_aa_substitutions.tsv",
    skip_rows=3,
    separator="\t",
).filter(pl.col("uniprot_id").is_in(set(df["uniprot"]))).rename(
    {"uniprot_id": "uniprot"}
).with_columns(
    pl.col("protein_variant").str.slice(0, 1).alias("aa")
).collect().write_csv(
    "AlphaMissense_aa_substitutions_filtered.tsv"
)
"""

am = (
    pl.read_csv( reference_database_folder / "AlphaMissense_aa_substitutions_filtered.tsv")
    .with_columns(
        pl.col("protein_variant")
        .str.slice(1, pl.col("protein_variant").str.len_chars() - 2)
        .cast(pl.Int64)
        .alias("pos")
    )
    .group_by(["uniprot", "pos"])
    .agg(pl.col("am_pathogenicity").mean())
)

df = df.join(other=am, on=["uniprot", "pos"], how="left")


# BIN FUNCTIONAL DATA INTO DISCRETE CATEGORIES
format = (
    df.select(
        [
            "uniprot",
            "residue",
            "pos",
            "ComplexName",
            "ComplexPortal_name",
            "Depth of Conservation (# of Organisms)",
            "pPSE",
            "am_pathogenicity",
            "structure_group",
            "reactivity_change"
        ]
    )
    .unique()
    .with_columns(
        pl.when(pl.col("ComplexName").is_not_null() | pl.col("ComplexPortal_name").is_not_null())
        .then(pl.lit("Complex"))
        .otherwise(pl.lit("Not complex"))
        .alias("Complex")
    )
    .drop("ComplexName")
    .drop("ComplexPortal_name")
    .drop_nulls()
    .with_columns(
        pl.when(pl.col("pPSE").gt(9))
        .then(pl.lit("buried"))
        .when(pl.col("pPSE").gt(5))
        .then(pl.lit("intermediate"))
        .otherwise(pl.lit("accessible"))
        .alias("pPSE"),
        pl.when(pl.col("Depth of Conservation (# of Organisms)").gt(49))
        .then(pl.lit("51+"))
        .when(pl.col("Depth of Conservation (# of Organisms)").gt(10))
        .then(pl.lit("11-50"))
        .otherwise(pl.lit("1-10"))
        .alias("Depth of Conservation (# of Organisms)"),
        # cutoffs defined in AlphaMissense supplement
        pl.when(pl.col("am_pathogenicity") > 0.564)
        .then(pl.lit("Pathogenic"))
        .when(pl.col("am_pathogenicity") < 0.34)
        .then(pl.lit("Benign"))
        .otherwise(pl.lit("Ambiguous"))
        .alias("am_class"),
    )
    .drop("am_pathogenicity")
)
# Read back by rc_visualization.Rmd for the alluvial plot; keep it in the repo.
format.write_csv(REPO_ROOT / "reactivity" / "formatted_cross_ref.csv")


print(len(df))

In [ ]:
"""
Species level conservation
"""

cysteine_conservation = (
    pl.scan_csv(
        reference_database_folder / "cysteine_conservation.csv",
        skip_rows=1,
    )
    .drop_nulls()
    .with_columns(
        pl.col(id_cols).str.split(" "),
        pl.col("Depth of Conservation (# of Organisms)")
        .truediv(1.02)  # 102 organisms
        .alias("percent conservation"),
    )
    .explode(id_cols)
    .sort(by="percent conservation")
    .with_columns(pl.concat_str(pl.lit("C"), pl.col("Cysteine")).alias("first_residue"))
    .filter(pl.col("Accession").is_in(set(df["uniprot"])))
    .group_by(["Accession", "first_residue"])
    .agg(pl.all().mean())
    .with_columns(pl.col("Accession").alias("uniprot"))
    .select(
        "uniprot",
        "first_residue",
        "Depth of Conservation (# of Organisms)",
        "mm10",
        "D. melanogaster",
        "C. elegans",
        "S. cerevisiae",
    )
    .collect()
)
df = (
    pl.read_csv(result_dir / "reactivity_changes_long_format.csv")
    .with_columns(pl.col("residue").str.split(",").list.get(0).alias("first_residue"))
    .join(other=cysteine_conservation, on=["uniprot", "first_residue"], how="left")
    .write_csv("species.csv")
)

In [ ]:
df = pl.read_csv(result_dir / "reactivity_changes_long_format.csv").with_columns(
    pl.col("residue")
    .str.split("C")
    .list.get(1)
    .str.split(",")
    .list.get(0)
    .cast(pl.Int64)
    .alias("pos")
).join(other=format, on=["uniprot", "pos"], how = "left").drop(
    ["residue_right"]
)

In [ ]:
conditions = [
    pl.col("reactivity_change"),
    pl.col("Complex").eq("Complex"),
    #pl.col("pPSE").eq("accessible"),
    pl.col("Depth of Conservation (# of Organisms)").eq("51+"),
    pl.col("am_class").eq("Pathogenic")
]

for i in range(1, len(conditions) + 1):
    print(f"Applying {i} condition(s)")
    filtered_df = df.filter(
        pl.all_horizontal(conditions[0:i])
    ).select(["uniprot", "residue", "protein"]).unique()
    print(f"Results: {filtered_df.shape[0]} residues")

In [ ]:
filtered_df.join(other = protein_residues, on = ["protein","residue"], how = "inner")

In [ ]:
df.filter(
    pl.col("protein").is_in(["EDC3", "GAPVD1", "ARHGAP45"]),
    pl.col("reactivity_change")
)

### ClinVar

In [ ]:
reactivity_data = pl.read_csv(result_dir / "reactivity_changes_long_format.csv").filter(pl.col("reactivity_change"))

study_proteins = set(reactivity_data["protein"])

clinvar_location = CLINVAR_TXT

protein_index_sep = "p."

get_protein_index = (
    pl.when(pl.col("Name").str.contains(protein_index_sep))
    .then(
        pl.col("Name")
        .str.split(protein_index_sep)
        .list.last()
        .str.split(")")
        .list.first()
        .str.extract(r"(\d+)", 1)
        .cast(pl.Int64)
        .alias("clinvar_position")
    )
    .otherwise(None)
)

# slow
df = (
    pl.scan_csv(
        clinvar_location, truncate_ragged_lines=True, separator="\t", ignore_errors=True
    )
    .filter(pl.col("GeneSymbol").is_in(study_proteins), pl.col("Assembly") == "GRCh37")
    .with_columns(get_protein_index)
    .collect()
)



In [ ]:
clinvar_residue_tolerance = 15

protein_residues = (
    reactivity_data.select(["protein", "residue"])
    .unique()
    .with_columns(
        pl.col("residue")
        .str.split("C")
        .list.get(1)
        .str.split(",")
        .list.get(0)
        .cast(pl.Int64)
        .alias("residue_position")
    )
    .join(other=df, left_on="protein", right_on="GeneSymbol")
    .with_columns(
        pl.col("residue_position")
        .sub(pl.col("clinvar_position"))
        .abs()
        .alias("residue_clinvar_distance")
    )
    .filter(
        pl.col("residue_clinvar_distance") < clinvar_residue_tolerance,
        pl.col("ClinicalSignificance").is_in(
            ["Likely pathogenic", "Pathogenic", "Pathogenic/Likely pathogenic"]
        ),
        # pl.col("protein").is_in(["TBCK", "PPP1R21"]),
    )
    .select(
        [
            "protein",
            "residue",
            "Type",
            "ClinicalSignificance",
            "PhenotypeList",
            "clinvar_position",
            "residue_clinvar_distance",
        ]
    )
    .group_by(["protein", "residue"])
    .agg(
        pl.all().unique().str.concat("|"),
        pl.col("residue_clinvar_distance").min().alias("min_residue_clinvar_distance"),
    )
)

protein_residues.write_csv(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/02_Figures/Figure 5_Reactivity into FERRY/Panels/clinvar/clinvar_reactivity_changes.csv"
)

## Phosphosite

Download with `wget http://www.phosphosite.org/downloads/Phosphorylation_site_dataset.gz`

In [ ]:
df = (
    pl.read_csv(result_dir / "reactivity_changes_long_format.csv")
    .with_columns(
        pl.col("residue")
        .str.split("C")
        .list.get(1)
        .str.split(",")
        .list.get(0)
        .cast(pl.Int64)
        .alias("pos")
    )
    .drop("location")
)

pl.scan_csv(
    reference_database_folder / "Phosphorylation_site_dataset",
    separator="\t",
    truncate_ragged_lines=True,
    skip_lines=3,
    ignore_errors=True,
).filter(
    pl.col("ORGANISM") == "human",
    pl.col("ACC_ID").is_in(set(df["uniprot"]))
).collect()

### Phosphoproteomics, reactivity, and WP intersection

In [ ]:
# Build gene-symbol -> UniProt accession map from the existing entry cache
gene_to_uniprot = build_gene_to_uniprot(cache)

# protein (gene symbol) -> uniprot + uniprot_function, all from cache.json / get_function
annotation = pl.DataFrame(
    {
        "protein": list(gene_to_uniprot.keys()),
        "uniprot": list(gene_to_uniprot.values()),
    }
).with_columns(
    pl.col("uniprot")
    .map_elements(lambda u: get_function(u, cache), return_dtype=pl.String)
    .alias("uniprot_function")
)


def annotate_uniprot(table):
    """Attach uniprot + uniprot_function columns to an omic_venn membership table."""
    return table.join(annotation, on="protein", how="left")


mpl.rcParams['pdf.fonttype'] = 42

def read_gmt_genes(path):
    """Return a flat list of all gene names in a GMT file (descriptions/URLs dropped)."""
    genes = []
    with open(path) as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) > 2:
                genes.extend(fields[2:])
    return genes


def omic_venn(filter_set):
    prefix = Path(
        "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/"
    )
    directions = ["Significant Up", "Significant Down"]
    files = [
        "02_reactivity/03_rc_analysis/reactivity_output/20251023_ratio_directionality/median_for_5+_peptides/output/reactivity_changes_long_format.csv",
        "05_phosphoproteomics/processed_results/expression-normalized_phosphorylation_table.csv",
        "01_unenriched proteomics/03_results/20250219_3reps/04_results/volcano_plots/volcano_data_long_format.csv",
    ]
    filters = [
        (pl.col("reactivity_change"), pl.col("condition").eq("TLR4")),
        pl.col("Regulation_phospho - TLR4 vs. M0 (9591 Residues)").is_in(directions),
        (pl.col("condition").eq("TLR4"), pl.col("Regulation").is_in(directions)),
    ]
    names = [
        "Reactivity changes (TLR4)",
        "Phosphorylated proteins in TLR4 (FC > 2, p < 0.05)",
        "Expression changes in TLR4 (FC > 1.5, p < 0.05)",
    ]
    sets = [
        set(
            pl.read_csv(prefix / files[i]).filter(
                filters[i], pl.col("protein").is_in(filter_set)
            )["protein"]
        )
        for i in range(3)
    ]


    fig, ax = plt.subplots(figsize=(2, 2))
    v = venn3(
        sets,
        set_labels=names,
        ax=ax,
        set_colors=["#FFCC31", "#4869B2", "green"],
    )
    venn3_circles(sets, ax=ax, linewidth=0.25)
    for text in [*v.set_labels, *v.subset_labels]:
        if text:
            text.set_fontsize(6)
            text.set_color("black")
            text.set_fontfamily("Arial")

    plt.tight_layout()
    # --- binary membership table ---
    all_proteins = sorted(set().union(*sets))
    table = pl.DataFrame({"protein": all_proteins}).with_columns(
        pl.col("protein").is_in(s).cast(pl.Int8).alias(name)
        for name, s in zip(names, sets)
    )
    return(fig, table)

# GO-term gene sets from MSigDB 2026.1.Hs (reference_dbs/msigdb_go_2026.csv, the same
# source as the R go_enrich() enrichment) so the venns and the enrichment use identical
# term memberships. Keyed by GO id.
_msigdb_go = pl.read_csv(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Vinogradova Laboratory/Henry_data processing/01_data_analysis_folders/06_macrophage/macrophage/reference_dbs/msigdb_go_2026.csv",
    columns=["GO_id", "term", "gene"],
)


def msigdb_go_genes(go_id):
    """Gene symbols of a MSigDB C5 GO set by GO id (matches go_enrich() membership)."""
    return set(_msigdb_go.filter(pl.col("GO_id") == go_id)["gene"])


def msigdb_go_term_genes(terms):
    """Union of gene symbols across MSigDB GO sets matched by (case-insensitive) term
    name. Mirrors go_genes_terms() in phosphoproteomics/visualization.Rmd so the venn
    uses the same multi-term gene set as the phospho volcano plot's highlighting."""
    tl = [t.lower() for t in terms]
    return set(_msigdb_go.filter(pl.col("term").str.to_lowercase().is_in(tl))["gene"])


output_dir = Path("/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/02_Figures/6-figure format_EV/Figure 3_Reactivity_part2_endocytosis/Panels/go_term_venn_diagram/")

endocytosis = msigdb_go_genes("GO:0006897")  # GOBP endocytosis

endosomal_transport = msigdb_go_genes("GO:0016197")  # GOBP endosomal transport

# Same GTPase gene set as the phospho volcano plot (gtpase_genes): union of these GO
# term names, not GO:0003924 alone. NB "gtpase regulator activity" has no MSigDB term
# and is silently dropped here just as it is in go_genes_terms().
gtpase = msigdb_go_term_genes([
    "gtpase activity", "gtpase activator activity", "gtpase regulator activity",
    "gtpase activating protein binding", "gtpase inhibitor activity", "gtpase binding",
])

vesicle_mediated = msigdb_go_genes("GO:0016192")  # GOBP vesicle-mediated transport



In [ ]:
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'

RED, BLACK = "#E8000B", "#000000"


def omic_change_sets(restrict=None):
    """{omic_name: set(proteins with a change)}; restrict to a gene set if given."""
    prefix = Path(
        "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/Manuscript 1/01_Data Analysis/01_Proteomics/"
    )
    directions = ["Significant Up", "Significant Down"]
    files = [
        "02_reactivity/03_rc_analysis/reactivity_output/20251023_ratio_directionality/median_for_5+_peptides/output/reactivity_changes_long_format.csv",
        "05_phosphoproteomics/processed_results/expression-normalized_phosphorylation_table.csv",
        "01_unenriched proteomics/03_results/20250219_3reps/04_results/volcano_plots/volcano_data_long_format.csv",
    ]
    filters = [
        [pl.col("reactivity_change"), pl.col("condition").eq("TLR4")],
        [pl.col("Regulation_phospho - TLR4 vs. M0 (9591 Residues)").is_in(directions)],
        [pl.col("condition").eq("TLR4"), pl.col("Regulation").is_in(directions)],
    ]
    names = [
        "Reactivity changes (TLR4)",
        "Phosphorylated proteins in TLR4 (FC > 2, p < 0.05)",
        "Expression changes in TLR4 (FC > 1.5, p < 0.05)",
    ]
    out = {}
    for i in range(3):
        conds = list(filters[i])
        if restrict is not None:
            conds.append(pl.col("protein").is_in(restrict))
        out[names[i]] = set(pl.read_csv(prefix / files[i]).filter(*conds)["protein"])
    return out


# denominators: total changes per omic, computed once
OMIC_TOTALS = {k: len(v) for k, v in omic_change_sets().items()}


def omic_fraction_bargraph(table, omic_totals, title="", figsize=None):
    """
    Per-omic: what % of that omic's total changes fall in this category.
    table: binary membership table from omic_venn (columns = omic names).
    omic_totals: {omic_name: total # changes in that omic}.
    """
    names = [c for c in table.columns if c != "protein"]
    hits = [int(table[n].sum()) for n in names]
    tots = [omic_totals[n] for n in names]
    pct  = [100 * h / t if t else 0 for h, t in zip(hits, tots)]

    y = list(range(len(names)))[::-1]
    if figsize is None:
        figsize = (3.2, 0.32 * len(names) + 0.6)
    fig, ax = plt.subplots(figsize=figsize)

    ax.barh(y, pct, color=RED, height=0.6, zorder=2)
    xmax = max(pct) * 1.35 if any(pct) else 1
    ax.set_xlim(0, xmax)
    for yi, p, h, t in zip(y, pct, hits, tots):
        ax.text(p + xmax * 0.02, yi, f"{p:.1f}%  ({h}/{t})",
                va="center", ha="left", fontsize=6, color="black")

    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=6)
    ax.set_xlabel("% of significant changes in profiling method", fontsize=6)
    ax.tick_params(labelsize=6, width=0.25, length=2)
    for s in ax.spines.values():
        s.set_linewidth(0.25)
    ax.spines[["top", "right"]].set_visible(False)
    if title:
        ax.set_title(title, fontsize=8, fontweight="bold")
    fig.tight_layout()
    return fig


def make_intersection(gene_set, set_name):
    fig, table = omic_venn(gene_set)
    
    plt.savefig(output_dir / f'{set_name}_venn.pdf', bbox_inches="tight")
    table.write_excel(output_dir / f"{set_name}_intesection_table.xlsx")

    bar = omic_fraction_bargraph(table, OMIC_TOTALS, title=set_name)
    bar.savefig(output_dir / f"{set_name}_omic_fraction_bargraph.pdf", bbox_inches="tight")
    table = annotate_uniprot(table)
    return fig, table

print(f"Writing to {output_dir}")
make_intersection(endocytosis, "Endocytosis")
make_intersection(set(endosomal_transport), "Endosomal transport")
make_intersection(gtpase, "GTPase activity")
make_intersection(vesicle_mediated, "Vesicle-mediated transport")

In [ ]:
import polars as pl
from pathlib import Path

go_dir = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/"
    "Manuscript 1/01_Data Analysis/01_Proteomics/00_reference lists/GO_term_lists"
)
phospho_path = Path(
    "/Users/henrysanford/Dropbox @RU Dropbox/Vinogradova Laboratory/Macrophage project/"
    "Manuscript 1/01_Data Analysis/01_Proteomics/05_phosphoproteomics/"
    "processed_results/expression-normalized_phosphorylation_table.csv"
)

# {column name to add : subfolder name on disk}  (note the hyphen in the first folder)
go_terms = {
    "Vesicle-mediated transport": "Vesicle-mediated transport",
    "GTPase activity": "GTPase activity",
    "Endosomal transport": "Endosomal transport",
}

df = pl.read_csv(phospho_path, infer_schema_length=2000)

for col_name, folder in go_terms.items():
    members = set(
        pl.read_csv(go_dir / folder / "02_genes_matching_query.csv")["genes"]
        .drop_nulls()
    )
    df = df.with_columns(pl.col("protein").is_in(members).alias(col_name))

df.write_csv(phospho_path.with_name("expression-normalized_phosphorylation_table_annotated.csv"))


In [ ]:
endosomal_transport = set(endosomal_transport)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib_venn import venn2, venn2_circles

mpl.rcParams['font.family'] = 'Arial'

fig, ax = plt.subplots(figsize=(2, 2))

v = venn2(
    [endosomal_transport, endocytosis],
    set_labels=('Endosomal transport', 'Endocytosis'),
    ax=ax,
    set_colors= ["#FFCC31", "#4869B2"]
)

# Outline circles at LW = 0.25
c = venn2_circles(
    [endosomal_transport, endocytosis],
    linewidth=0.25,
    ax=ax,
)

# All labels (set names + subset counts) in Arial size 6
for t in v.set_labels:
    if t is not None:
        t.set_fontfamily('Arial')
        t.set_fontsize(6)
for t in v.subset_labels:
    if t is not None:
        t.set_fontfamily('Arial')
        t.set_fontsize(6)

fig.tight_layout()
plt.savefig(output_dir / "endocytosis_endosomal_tranposrt.pdf")
plt.show()


In [ ]:
table.filter(pl.col(name).eq(1) for name in names)